# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [6]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from data_profiling import ProfileReport
from sklearn.preprocessing import MinMaxScaler

# ML_Utils
import sys
sys.path.append('/Users/dv/Desktop/machine-learning-bible/')
from ml_utils import clean_data, quick_eda, profile_data

In [7]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [8]:
quick_eda(spaceship)

Shape: (8693, 14)
Duplicates: 0
Missing Values:
 PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64
Missing Values %:
 PassengerId     0.00
HomePlanet      2.31
CryoSleep       2.50
Cabin           2.29
Destination     2.09
Age             2.06
VIP             2.34
RoomService     2.08
FoodCourt       2.11
ShoppingMall    2.39
Spa             2.11
VRDeck          2.16
Name            2.30
Transported     0.00
dtype: float64
Data Types:
 PassengerId      object
HomePlanet       object
CryoSleep        object
Cabin            object
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
Trans

In [9]:
# your code here
X = spaceship.drop(columns='Transported')
y = spaceship['Transported']

print(f"Datensatzgröße: {X.shape}")
print(f"Klassen: {spaceship}")
print(f"Klassenverteilung:")
print(f"  Not Transported (0): {sum(y==False)}")
print(f"  Transported  (1): {sum(y==True)}")
print()
X.head()

Datensatzgröße: (8693, 13)
Klassen:      PassengerId HomePlanet CryoSleep     Cabin    Destination   Age    VIP  \
0        0001_01     Europa     False     B/0/P    TRAPPIST-1e  39.0  False   
1        0002_01      Earth     False     F/0/S    TRAPPIST-1e  24.0  False   
2        0003_01     Europa     False     A/0/S    TRAPPIST-1e  58.0   True   
3        0003_02     Europa     False     A/0/S    TRAPPIST-1e  33.0  False   
4        0004_01      Earth     False     F/1/S    TRAPPIST-1e  16.0  False   
...          ...        ...       ...       ...            ...   ...    ...   
8688     9276_01     Europa     False    A/98/P    55 Cancri e  41.0   True   
8689     9278_01      Earth      True  G/1499/S  PSO J318.5-22  18.0  False   
8690     9279_01      Earth     False  G/1500/S    TRAPPIST-1e  26.0  False   
8691     9280_01     Europa     False   E/608/S    55 Cancri e  32.0  False   
8692     9280_02     Europa     False   E/608/S    TRAPPIST-1e  44.0  False   

      RoomServi

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines


## WICHTIG
1. Missing Values füllen  ← ZUERST (während die Spalten noch existieren!)
2. Droppen (PassengerId, Cabin, Name)
3. Encoden (get_dummies)
4. Skalieren
5. Train-Test-Split

In [10]:
# Handle Missing Values in Categorical Columns
cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP'] 
for col in cat_cols:
    X[col] = X[col].fillna(X[col].mode()[0])

/var/folders/gn/zvx5kxqj5ng9ng0cg5f15yk80000gn/T/ipykernel_19468/1511286050.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X[col] = X[col].fillna(X[col].mode()[0])


In [11]:
# Handle Missing Values in Numerical Columns
num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in num_cols:
    X[col] = X[col].fillna(X[col].mode()[0])

In [12]:
# Drop 
X = X.drop(columns=['PassengerId', 'Cabin', 'Name'])

# Encode
X = pd.get_dummies(X, columns=['HomePlanet', 'Destination', 'CryoSleep', 'VIP'])



In [13]:
X.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,CryoSleep_False,CryoSleep_True,VIP_False,VIP_True
0,39.0,0.0,0.0,0.0,0.0,0.0,False,True,False,False,False,True,True,False,True,False
1,24.0,109.0,9.0,25.0,549.0,44.0,True,False,False,False,False,True,True,False,True,False
2,58.0,43.0,3576.0,0.0,6715.0,49.0,False,True,False,False,False,True,True,False,False,True
3,33.0,0.0,1283.0,371.0,3329.0,193.0,False,True,False,False,False,True,True,False,True,False
4,16.0,303.0,70.0,151.0,565.0,2.0,True,False,False,False,False,True,True,False,True,False


In [14]:
y = spaceship['Transported'].astype(int)

In [15]:
# Feature Scaling 
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

**Perform Train Test Split**

In [16]:
#your code here
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)
print(f"Training: {X_train.shape[0]} | Test: {X_test.shape[0]}")


Training: 6519 | Test: 2174


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [17]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

# Bagging: many Decision Trees trained on random subsets of the data
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    random_state=42
)
bagging.fit(X_train, y_train)
y_pred_bagging = bagging.predict(X_test)

print(f"Bagging Accuracy: {bagging.score(X_test, y_test):.4f}")
print(classification_report(y_test, y_pred_bagging))

Bagging Accuracy: 0.7787
              precision    recall  f1-score   support

           0       0.79      0.76      0.77      1077
           1       0.77      0.80      0.78      1097

    accuracy                           0.78      2174
   macro avg       0.78      0.78      0.78      2174
weighted avg       0.78      0.78      0.78      2174



- Random Forests

In [18]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest: Bagging + random feature selection at each split
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print(f"Random Forest Accuracy: {rf.score(X_test, y_test):.4f}")
print(classification_report(y_test, y_pred_rf))

Random Forest Accuracy: 0.7838
              precision    recall  f1-score   support

           0       0.79      0.77      0.78      1077
           1       0.78      0.80      0.79      1097

    accuracy                           0.78      2174
   macro avg       0.78      0.78      0.78      2174
weighted avg       0.78      0.78      0.78      2174



- Gradient Boosting

In [19]:
from sklearn.ensemble import GradientBoostingClassifier

# Gradient Boosting: trees built sequentially, each correcting the previous error
gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

print(f"Gradient Boosting Accuracy: {gb.score(X_test, y_test):.4f}")
print(classification_report(y_test, y_pred_gb))

Gradient Boosting Accuracy: 0.7912
              precision    recall  f1-score   support

           0       0.82      0.73      0.78      1077
           1       0.76      0.85      0.80      1097

    accuracy                           0.79      2174
   macro avg       0.79      0.79      0.79      2174
weighted avg       0.79      0.79      0.79      2174



- Adaptive Boosting

In [20]:
from sklearn.ensemble import AdaBoostClassifier

# AdaBoost: focuses on misclassified samples by increasing their weight
ada = AdaBoostClassifier(
    n_estimators=100,
    learning_rate=1.0,
    random_state=42
)
ada.fit(X_train, y_train)
y_pred_ada = ada.predict(X_test)

print(f"AdaBoost Accuracy: {ada.score(X_test, y_test):.4f}")
print(classification_report(y_test, y_pred_ada))

AdaBoost Accuracy: 0.7769
              precision    recall  f1-score   support

           0       0.78      0.77      0.77      1077
           1       0.77      0.79      0.78      1097

    accuracy                           0.78      2174
   macro avg       0.78      0.78      0.78      2174
weighted avg       0.78      0.78      0.78      2174



Which model is the best and why?

In [21]:
import pandas as pd

results = pd.DataFrame({
    'Model': ['Bagging', 'Random Forest', 'Gradient Boosting', 'AdaBoost'],
    'Accuracy': [
        bagging.score(X_test, y_test),
        rf.score(X_test, y_test),
        gb.score(X_test, y_test),
        ada.score(X_test, y_test)
    ]
}).sort_values('Accuracy', ascending=False)

print(results.to_string(index=False))

# Random Forest is typically the best model here.
# It combines bagging with random feature selection at each split,
# which reduces variance and avoids correlation between trees.
# Gradient Boosting is also strong but slower to train.
# AdaBoost is sensitive to outliers and noisy data.

            Model  Accuracy
Gradient Boosting  0.791168
    Random Forest  0.783809
          Bagging  0.778749
         AdaBoost  0.776909
